# 09 — Difference-in-Differences: From 2×2 to the Modern Toolkit
**References:** Baker, Callaway, Cunningham, Goodman-Bacon & Sant'Anna (2026) "Difference-in-Differences Designs: A Practitioner's Guide" (*JEL* 64(2)) · Roth, Sant'Anna, Bilinski & Poe (2023, *J. Econometrics*) · Card & Krueger (1994) · Sant'Anna & Zhao (2020) · Callaway & Sant'Anna (2021) · Goodman-Bacon (2021) · Rambachan & Roth (2023) · Mixtape Sessions *Advanced DiD* (Roth) & *Frontiers in DiD* (Callaway)

## Narrative thread
```
2x2 building block -> Forward engineering -> Covariates (RA/IPW/DR) -> Event studies -> Pre-trend pitfalls -> Honest sensitivity -> Staggered ATT(g,t)
```

## The 2×2 design

Treated and control groups, before and after treatment. Neither single comparison works:
- After − before (treated only): contaminated by **time trends**
- Treated − control (after only): contaminated by **group differences**

DiD subtracts out both:
$$\hat\tau_{DD} = \big(\bar Y_{T,post} - \bar Y_{T,pre}\big) - \big(\bar Y_{C,post} - \bar Y_{C,pre}\big)$$

The estimand is the **ATT** (not the ATE), and its identification rests on two named
assumptions (following the notation of Baker et al. 2026):

**Assumption NA (No Anticipation):** $Y_{i,t}(1) = Y_{i,t}(0)$ for all treated units in all
pre-treatment periods — the treatment has no effect before it starts. (If announcement
effects exist, "treatment" begins at announcement.)

**Assumption PT (Parallel Trends):** the average change in *untreated* potential outcomes
is the same in both groups,
$$E[Y_{t=2}(0) - Y_{t=1}(0) \mid D=1] = E[Y_{t=2}(0) - Y_{t=1}(0) \mid D=0]$$

Untestable (it is about the counterfactual). Note it is *weaker* than random assignment —
groups may differ in levels — but it is **functional-form sensitive**: PT in levels and PT
in logs are different assumptions (Roth & Sant'Anna 2023b give a falsification test).

## Forward engineering (the organizing idea of Baker et al. 2026)

Complex DiD designs should be built like this:
```
1. define the target parameter (which ATT? weighted how?)
2. state the identifying assumptions (which PT? NA? overlap?)
3. derive the estimator from 1 + 2      <- "forward engineering"
```
not by running a familiar regression and asking afterwards what it estimates
("reverse engineering"). Every design below — covariates, event studies, staggered
timing — is an **aggregation of 2×2 building blocks**, each valid under its own explicit
PT assumption.

Two practical notes from the paper's Medicaid running example:
- **Weights define the parameter.** The unweighted (county-level) 2×2 DiD gave $+0.1$
  deaths per 100k while the population-weighted one gave $-2.6$: not a robustness check,
  but two *different* target parameters (effect on the average county vs. average adult).
- The canonical **Card & Krueger (1994)** design (NJ minimum wage vs. PA fast food) is the
  2×2 template simulated below.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── 2x2 DiD, Card-Krueger style ───────────────────────────────────────────
n_per = 300
periods = np.arange(-4, 4)                        # 4 pre, 4 post periods
TRUE_EFFECT = 3.0

rows = []
for g, (group, base, trend) in enumerate([('control', 20.0, 0.8), ('treated', 26.0, 0.8)]):
    for t in periods:
        eff = TRUE_EFFECT if (group == 'treated' and t >= 0) else 0.0
        y = base + trend*t + eff + np.random.normal(0, 2.0, n_per)
        rows.append(pd.DataFrame({'group': group, 't': t, 'y': y}))
df = pd.concat(rows, ignore_index=True)
df['treat'] = (df.group == 'treated').astype(int)
df['post']  = (df.t >= 0).astype(int)

# Manual 2x2 ("four means" — eq. (6) of Baker et al. 2026)
m = df.groupby(['group', 'post'])['y'].mean().unstack()
dd = (m.loc['treated', 1] - m.loc['treated', 0]) - (m.loc['control', 1] - m.loc['control', 0])

# Regression version: numerically identical, easy clustered SEs
reg = smf.ols('y ~ treat * post', df).fit()
print(f'True effect:              {TRUE_EFFECT:.2f}')
print(f'Manual double difference: {dd:.3f}')
print(f"Regression treat:post:    {reg.params['treat:post']:.3f}  (SE {reg.bse['treat:post']:.3f})")

means = df.groupby(['group', 't'])['y'].mean().unstack(0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(means.index, means['treated'], 'o-', color='#d62728', label='treated')
ax.plot(means.index, means['control'], 'o-', color='#1f77b4', label='control')
cf = means.loc[-1, 'treated'] + (means['control'] - means.loc[-1, 'control'])
ax.plot(means.index[means.index >= -1], cf[means.index >= -1], '--', color='#d62728',
        alpha=0.6, label='treated counterfactual (parallel trends)')
ax.axvline(-0.5, color='k', ls=':', lw=1)
ax.set_xlabel('period (treatment at t = 0)'); ax.set_ylabel('mean outcome')
ax.set_title('DiD: the control group supplies the counterfactual trend')
ax.legend(); plt.tight_layout(); plt.show()

## Covariates: conditional parallel trends and doubly robust DiD

When groups differ in covariates that drive outcome *trends* (not just levels),
unconditional PT fails. Baker et al. (2026, §4) replace it with:

**Assumption CPT (Conditional Parallel Trends):** PT holds within each covariate stratum,
$$E[\Delta Y(0) \mid X, D=1] = E[\Delta Y(0) \mid X, D=0]$$

**Assumption SO (Strong Overlap):** $\epsilon < P(D=1 \mid X) < 1 - \epsilon$.

### Why "just add $X$ to the TWFE regression" fails

The popular specification $Y_{it} = \theta_t + \eta_i + \beta D_{it} + X_{it}'\gamma + e_{it}$
has three problems (Caetano & Callaway 2024):
1. **time-invariant $X$ drops out** — you must interact $X$ with time to control trends;
2. with heterogeneous effects, $\beta$ is a **non-convexly weighted** average of
   $ATT_X$'s plus bias terms — it can even have the wrong sign;
3. **time-varying covariates** affected by treatment are bad controls (notebook 04).

### Forward-engineered estimators of the ATT under CPT + SO

| Estimator | Model needed | Recipe |
|---|---|---|
| RA (regression adjustment) | outcome: $\mu(x) = E[\Delta Y \mid X{=}x, D{=}0]$ | $\frac{1}{n_1}\sum_{D=1} (\Delta Y_i - \hat\mu(X_i))$ |
| IPW (Abadie 2005) | propensity: $p(x) = P(D{=}1\mid x)$ | reweight controls by $\frac{p(X)}{1-p(X)}$, compare means of $\Delta Y$ |
| **DR** (Sant'Anna & Zhao 2020) | either one | $\frac{1}{n_1}\sum_{D=1}\!\big(\Delta Y_i - \hat\mu(X_i)\big) - \sum_{D=0} \hat w_i \big(\Delta Y_i - \hat\mu(X_i)\big)$ |

The **doubly robust** combination is consistent if *either* working model is right and is
the paper's recommended default (it is what the `did` / `DRDID` packages implement, with
propensity scores trimmed at 0.995). Always plot the two groups' propensity-score
distributions to check overlap before trusting any of them.


In [ ]:
# ── RA vs IPW vs DR when only CONDITIONAL parallel trends holds ──────────
# Two periods. x drives BOTH selection into treatment and the outcome trend
# (nonlinearly), so unconditional PT fails and linear-in-x models are wrong.
from sklearn.linear_model import LogisticRegression

def did_estimators(n=4000, seed=0):
    rng = np.random.default_rng(seed)
    x  = rng.normal(0, 1, n)
    p  = 1/(1 + np.exp(-0.9*x))
    d  = rng.binomial(1, p)
    trend = 1.0 + 2.0*x + 1.5*x**2               # untreated trend depends on x
    tau   = 2.0 + 1.0*x                           # heterogeneous effect
    dy = trend + tau*d + rng.normal(0, 1, n)      # observed Delta Y
    att_true = tau[d == 1].mean()

    out = {}
    out['Naive 2x2 (uncond. PT)'] = dy[d==1].mean() - dy[d==0].mean()
    # "Add x to the regression" (linear TWFE-style in differences)
    out['OLS  dY ~ D + x'] = smf.ols('dy ~ d + x', pd.DataFrame(dict(dy=dy, d=d, x=x))).fit().params['d']
    # RA with a misspecified (linear) outcome model
    b = np.polyfit(x[d==0], dy[d==0], 1)
    out['RA, outcome model WRONG'] = (dy[d==1] - np.polyval(b, x[d==1])).mean()
    # RA with the correct (quadratic) outcome model
    b2 = np.polyfit(x[d==0], dy[d==0], 2)
    mu_hat = np.polyval(b2, x)
    out['RA, outcome model right'] = (dy[d==1] - mu_hat[d==1]).mean()
    # IPW (Abadie 2005), Hajek version, correctly specified logit
    e = np.clip(LogisticRegression(C=1e6).fit(x.reshape(-1,1), d).predict_proba(x.reshape(-1,1))[:,1], 1e-3, 0.995)
    w0 = (e/(1-e))[d==0]
    out['IPW, PS model right'] = dy[d==1].mean() - np.average(dy[d==0], weights=w0)
    # DR (Sant'Anna & Zhao 2020): WRONG linear outcome model + right PS
    mu_lin = np.polyval(b, x)
    out['DR (wrong outcome + right PS)'] = ((dy[d==1] - mu_lin[d==1]).mean()
                                            - np.average(dy[d==0] - mu_lin[d==0], weights=w0))
    return out, att_true

reps = [did_estimators(seed=s) for s in range(200)]
att_true = np.mean([t for _, t in reps])
print(f'True ATT = {att_true:.3f}   (mean over {len(reps)} simulations)\n')
print(f'{"estimator":<32} {"mean est":>9} {"bias":>8}')
for k in reps[0][0]:
    v = np.mean([r[k] for r, _ in reps])
    print(f'{k:<32} {v:9.3f} {v - att_true:8.3f}')
print('\nOnly estimators that model the x-specific trend correctly — or the DR')
print('combination with ONE correct working model — recover the ATT.')

## Event studies: dynamics and pre-trends in one picture

With several periods, estimate a coefficient per relative time (base period $t = -1$,
i.e. $g-1$):
$$Y_{it} = \alpha_i + \lambda_t + \sum_{k \neq -1} \tau_k\, \mathbb{1}\{t - E_i = k\} + \varepsilon_{it}$$

Identification of the post coefficients requires **PT-ES**: parallel trends in *every*
post-treatment period — long-run effects need stronger assumptions than short-run ones.
Two reporting cautions from Baker et al. (2026, §5.1):

- **Cluster SEs at the unit level** (Bertrand, Duflo & Mullainathan 2004), and use
  **simultaneous (sup-t) confidence bands** when interpreting the whole path — pointwise
  bands understate joint uncertainty (Montiel Olea & Plagborg-Møller 2019; built into the
  `did` package).
- The popular shortcut of a second static regression on $D_{it}$ (collapse to one
  coefficient) does **not** equal the average of post-period $\widehat{ATT}(t)$'s: it also
  subtracts the average *pre*-period estimate. In the paper's application the two differ
  by a factor of ~3.6 ($-0.70$ vs $-2.53$). Aggregate the building blocks instead.


In [ ]:
# ── Event-study regression on the simulated panel ────────────────────────
df['rel'] = np.where(df.treat == 1, df.t, -1)      # controls held at base period
dummies = pd.get_dummies(df.rel, prefix='k', dtype=float).drop(columns=['k_-1'])
X = pd.concat([dummies, pd.get_dummies(df.t, prefix='t', dtype=float, drop_first=True),
               df[['treat']]], axis=1)
res = sm.OLS(df.y, sm.add_constant(X)).fit()

ks = sorted([int(c.split('_')[1]) for c in dummies.columns])
coef = [res.params[f'k_{k}'] for k in ks]
ci   = [1.96*res.bse[f'k_{k}'] for k in ks]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.errorbar(ks, coef, yerr=ci, fmt='o', capsize=3, color='#1f77b4')
ax.axhline(0, color='k', lw=1); ax.axvline(-0.5, color='k', ls=':', lw=1)
ax.axhline(TRUE_EFFECT, color='#d62728', ls='--', lw=1, label='true post effect')
ax.set_xlabel('periods relative to treatment'); ax.set_ylabel(r'$\tau_k$')
ax.set_title('Event study: flat pre-trends, jump at adoption')
ax.legend(); plt.tight_layout(); plt.show()

## Pre-trends: three lessons (Roth 2022; Baker et al. 2026, §5.1.2)

1. **Pre-trends are not the assumption.** PT restricts *post*-treatment untreated
   trends; pre-trends measure the "wrong" periods. Flat pre-trends neither prove PT nor
   do sloped ones always refute it (e.g., selection on shocks far before treatment).
2. **Pre-tests are low-powered.** Tests of $\tau_{-k} = 0$ routinely fail to detect
   violations large enough to overturn the conclusions — and **conditioning the analysis
   on having passed a pre-test distorts inference further** (Roth 2022): passing draws are
   selected, so bias and undercoverage survive the screening.
3. **Replace the eye-test with sensitivity analysis** (Rambachan & Roth 2023): bound how
   large post-treatment PT violations could be — e.g. at most $\bar M$ times the largest
   pre-period violation ("relative magnitudes") — and report the *identified set* and the
   **breakdown value** $\bar M^*$ at which the conclusion dies. In the Medicaid example,
   allowing violations as big as the largest pre-trend widens the CI to $[-11.1, 5.1]$ —
   the pre-trends simply do not carry much information.


In [ ]:
# ── Lesson 2 quantified: pre-tests are low-powered AND screening distorts ─
# DGP: NO true effect; treated group drifts by delta per period (PT violated).
def one_did(delta, n=40, n_pre=4, n_post=4, seed=None):
    rng = np.random.default_rng(seed)
    ts = np.arange(-n_pre, n_post)
    # unit-period panel, difference out unit effects by working with group means
    yT = delta*(ts + 1) + rng.normal(0, 1.2/np.sqrt(n), len(ts))  # treated group mean
    yC = rng.normal(0, 1.2/np.sqrt(n), len(ts))                    # control group mean
    gap = yT - yC
    gap = gap - gap[n_pre - 1]                    # normalize at k = -1
    se_gap = np.sqrt(2) * 1.2/np.sqrt(n) * np.sqrt(2)              # diff of 2 means, minus base
    pre_t  = gap[:n_pre - 1] / se_gap             # t-stats of pre coefficients
    passed = np.all(np.abs(pre_t) < 1.96)         # "no significant pre-trend"
    est_post = gap[n_pre]                          # event-time-0 estimate (true effect = 0)
    covered = abs(est_post) <= 1.96*se_gap
    return passed, est_post, covered

delta = 0.15                                       # modest linear PT violation
runs = [one_did(delta, seed=s) for s in range(4000)]
passed = np.array([r[0] for r in runs]); est = np.array([r[1] for r in runs])
cov = np.array([r[2] for r in runs])

print(f'True effect = 0; PT violated by a linear drift of {delta}/period')
print(f'Pre-test power (share of datasets where the violation is DETECTED): {1 - passed.mean():.2f}')
print(f'Bias at event time 0, all datasets:            {est.mean():+.3f}')
print(f'Bias conditional on PASSING the pre-test:      {est[passed].mean():+.3f}')
print(f'95% CI coverage conditional on passing:        {cov[passed].mean():.2f}  (nominal 0.95)')
print('\nMost violated datasets pass the pre-test, and among those the estimate')
print('is still biased with undercovering CIs — "pretest with caution" (Roth 2022).')

In [ ]:
# ── Lesson 3: sensitivity in the spirit of Rambachan & Roth (2023) ───────
# Relative-magnitudes bounds applied to OUR event study from the cell above:
# allow post-period PT violations up to Mbar x (largest pre-period coefficient).
pre_ks  = [k for k in ks if k < -1]
post_k0 = ks.index(0)
gamma_max = max(abs(res.params[f'k_{k}']) for k in pre_ks)      # largest pre-trend
beta0, se0 = coef[post_k0], res.bse['k_0']

breakdown = (beta0 - 1.96*se0) / gamma_max   # Mbar where the robust interval hits 0
Mbars = np.linspace(0, 1.3*breakdown, 61)
lo = beta0 - 1.96*se0 - Mbars*gamma_max      # conservative robust interval
hi = beta0 + 1.96*se0 + Mbars*gamma_max

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.fill_between(Mbars, lo, hi, alpha=0.25, color='#1f77b4', label='robust interval for ATT(0)')
ax.axhline(0, color='k', lw=1)
ax.axvline(1.0, color='#888', ls=':', lw=1)
ax.axvline(breakdown, color='#d62728', ls='--', lw=1.5,
           label=f'breakdown value $\\bar{{M}}^* \\approx$ {breakdown:.2f}')
ax.set_xlabel(r'$\bar M$ = allowed violation / largest pre-trend coefficient')
ax.set_ylabel('ATT at event time 0')
ax.set_title('Honest-DiD-style sensitivity: how big a PT violation kills the result?')
ax.legend()
plt.tight_layout(); plt.show()
print(f'Largest pre-period coefficient: {gamma_max:.3f}   ATT(0) = {beta0:.3f} (SE {se0:.3f})')
print(f'The effect survives violations up to ~{breakdown:.1f}x the observed pre-trends.')
print('(The real Rambachan-Roth method uses moment inequalities — R package HonestDiD.)')

## Staggered adoption: ATT(g,t) building blocks

With cohorts adopting at different dates $g$, define **group-time effects**
(Callaway & Sant'Anna 2021):
$$ATT(g, t) = E[Y_{i,t}(g) - Y_{i,t}(\infty) \mid G_i = g]$$

Each one is identified by a plain 2×2 comparison; the choice of *comparison group* is the
choice of parallel-trends assumption (Baker et al. 2026, §5.2):

| Assumption | Comparison group for $ATT(g,t)$ | Trade-off |
|---|---|---|
| **PT-GT-NEV** | never-treated units | cleanest; needs a credible never-treated pool |
| **PT-GT-NYT** | not-yet-treated by $t$ | more data; the paper's preferred middle ground |
| **PT-GT-ALL** | all periods & groups | most precision; *imposes* parallel pre-trends |

Estimators: sample analogs of the 2×2 blocks (Callaway & Sant'Anna 2021 — eq. 27/28 of
the paper), a saturated interaction regression (Sun & Abraham 2021), or "extended TWFE" /
imputation under PT-GT-ALL (Wooldridge 2025 = Borusyak-Jaravel-Spiess 2024 = Gardner 2022
with balanced panels). Then **aggregate** the $ATT(g,t)$'s into an event study
$ATT_{es}(e) = \sum_g w_g\, ATT(g, g+e)$ with weights = each cohort's share of treated
units observed at event time $e$.

### Why not static TWFE?

$Y_{it} = \eta_i + \theta_t + \beta^{twfe} D_{it} + e_{it}$ implicitly uses
**already-treated units as controls**. In the simplest 3-group example the estimand is
$$\beta^{twfe} = ATT(2,2) + w_1\,\big[ATT(1,1) - ATT(1,2)\big]$$
— a *negative* weight on $ATT(1,2)$ unless effects are static. With dynamic effects
$\beta^{twfe}$ can flip sign (Goodman-Bacon 2021; de Chaisemartin & D'Haultfœuille 2020).
Baker et al. (2026): "*we do not recommend using it*."


In [ ]:
# ── Callaway-Sant'Anna ATT(g,t) with a never-treated group ────────────────
# Three groups: never-treated, early cohort (g=4), late cohort (g=8).
# Effects are dynamic AND heterogeneous across cohorts.
n_g, T = 100, 12
groups = {np.inf: 0.0, 4: 1.0, 8: 0.5}            # cohort -> effect slope per period
rows = []
uid = 0
for g_, slope in groups.items():
    fe = np.random.normal(0, 2, n_g)
    for i in range(n_g):
        for t in range(T):
            e_time = t - g_ if np.isfinite(g_) else -99
            tau = slope * (e_time + 1) if (np.isfinite(g_) and t >= g_) else 0.0
            rows.append({'unit': uid, 'g': g_, 't': t, 'd': int(np.isfinite(g_) and t >= g_),
                         'y': fe[i] + 0.5*t + tau + np.random.normal(0, 1), 'tau': tau})
        uid += 1
panel = pd.DataFrame(rows)

# ATT(g,t) via eq. (28): never-treated comparison, base period g-1
gm = panel.groupby(['g', 't'])['y'].mean()
att_gt = {}
for g_ in [4, 8]:
    for t in range(2, T):                          # include t < g pre-trend estimates
        att_gt[(g_, t)] = ((gm.loc[(g_, t)] - gm.loc[(g_, g_ - 1)])
                           - (gm.loc[(np.inf, t)] - gm.loc[(np.inf, g_ - 1)]))

# Event-study aggregation ATT_es(e), weights = cohort share among treated at e
att_es, true_es = {}, {}
for e_ in range(0, T - 4):
    cs = [g_ for g_ in [4, 8] if g_ + e_ < T]      # cohorts observed at event time e
    att_es[e_]  = np.mean([att_gt[(g_, g_ + e_)] for g_ in cs])   # equal cohort sizes
    true_es[e_] = np.mean([groups[g_] * (e_ + 1) for g_ in cs])

# TWFE for contrast
def demean(s, by): return s - s.groupby(by).transform('mean')
yd  = demean(demean(panel.y, panel.unit), panel.t)
dd_ = demean(demean(panel.d.astype(float), panel.unit), panel.t)
twfe = (dd_ @ yd) / (dd_ @ dd_)
true_att = panel.loc[panel.d == 1, 'tau'].mean()

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for g_, color in [(4, '#d62728'), (8, '#1f77b4')]:
    es = {t - g_: v for (gg, t), v in att_gt.items() if gg == g_}
    axes[0].plot(list(es), list(es.values()), 'o-', ms=4, color=color, label=f'cohort g={g_}')
axes[0].axhline(0, color='k', lw=1); axes[0].axvline(-0.5, color='k', ls=':', lw=1)
axes[0].set_title('ATT(g, t) in event time, never-treated controls')
axes[0].set_xlabel('event time e = t - g'); axes[0].legend()

es_e = list(att_es)
axes[1].plot(es_e, [att_es[e_] for e_ in es_e], 'o-', color='#2ca02c', label='aggregated $ATT_{es}(e)$')
axes[1].plot(es_e, [true_es[e_] for e_ in es_e], 'k--', lw=1, label='true (same composition)')
axes[1].axhline(twfe, color='#ff7f0e', ls='--', label=f'static TWFE = {twfe:.2f}')
axes[1].set_title('Aggregation vs the TWFE single number')
axes[1].set_xlabel('event time e'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'True ATT over treated observations: {true_att:.2f}')
print(f'Static TWFE:                        {twfe:.2f}   <- shrunk by forbidden comparisons')
print(f'Mean of aggregated ATT_es(e):       {np.mean(list(att_es.values())):.2f}'
      f'   (true mean, same composition: {np.mean(list(true_es.values())):.2f})')
pre_max = max(abs(v) for (g_, t), v in att_gt.items() if t < g_)
print(f'Largest pre-treatment ATT(g,t):     {pre_max:.2f}   (placebo check ~ 0)')
print('\nNote the jump in ATT_es(e) at e = 4: the late cohort runs out of data and the')
print('composition changes — "imbalance in event time" (Baker et al. 2026, eq. 32);')
print('impose balance in event time if a constant composition matters.')

## The Baker et al. (2026) practitioner checklist

1. **Define target parameters** — which ATT, over whom, weighted how (units vs population)?
2. **State the identification assumptions** — which PT (unconditional / CPT / NEV / NYT /
   ALL), no anticipation, overlap. Say it explicitly so readers can debate it.
3. **Determine the estimation method** — sample means for simple designs; RA / IPW /
   doubly robust for conditional designs; never let the regression pick the estimand.
4. **Discuss sources of uncertainty** — what is random, cluster at the level of treatment
   assignment/sampling; simultaneous bands for event studies.
5. **Estimate.**
6. **Conduct sensitivity analysis** — Rambachan-Roth bounds, functional-form checks
   (Roth & Sant'Anna 2023b), placebo timing.
7. **Conduct heterogeneity analysis** — $ATT(g,t)$, event time, covariate partitions.
8. **Keep learning** — DiD is one design among many; if its assumptions are implausible,
   change designs, not standards.

## Key takeaways

| Concept | Statement |
|---|---|
| 2×2 DiD | Four means; identified by NA + PT; estimand is the ATT |
| Forward engineering | Parameter → assumptions → estimator, never the reverse |
| Weights | Weighted vs unweighted are different *parameters*, not robustness checks |
| Covariates | CPT + overlap; use RA / IPW / **DR** (Sant'Anna-Zhao), not "add X to TWFE" |
| Pre-trends | Not the assumption; low-powered; screening on them distorts inference |
| Honest DiD | Report Rambachan-Roth robust sets and the breakdown $\bar M^*$ |
| Staggered | Estimate $ATT(g,t)$ with clean (never/not-yet-treated) controls, then aggregate |
| TWFE | Negative weights on forbidden comparisons — "we do not recommend using it" |
